================================================================================
模块 02：市场微观结构与订单簿动力学 · 第 3/5 课
第 3 课：订单流不平衡 (Order Flow Imbalance, OFI) 与高频价格预测
================================================================================

::: {.callout-note}
### 投决会高频第一性原理：价格跳跃不是成交量的线性函数，而是挂单簿净通量的不平衡释放
- **[微观驱动公理]**：已成交量（Trade Volume）是事后发生的滞后历史数据，而盘口静态挂单深度充斥着虚假远端挂单。Cont, Kukanov and Stoikov (2014) 指出，短期价格变动的真正推手是相邻离散快照时刻之间，最优买卖档位由新挂单、吃单与主动撤单综合产生的净能量流（Net Flux）。
- **[OFI 的代数守恒]**：订单流不平衡（Order Flow Imbalance, OFI）将买方盘口净驱动力与卖方净驱动力进行正交差分，彻底消除了盘口非对称单位量纲，输出纯净的微观价格跳跃先行信号。
- **[Kyle's Lambda 微观映射]**：在秒级与 Tick 级窗口内，中间价变动量与 OFI 存在极高的线性决定系数（$R^2$ 常达 60%~80%），其回归斜率直接对应微观市场冲击成本弹性。
:::

- **[Cont-Kukanov-Stoikov (2014) OFI 严格代数推导]**：
  - 设连续两个快照时刻 $t-1$ 与 $t$，最优买盘报价量为 $(P^b_t, q^b_t)$，最优卖盘报价量为 $(P^a_t, q^a_t)$。
  - 买方订单流驱动项 $I^b_t$ 分三态严格判定：

$$
I^b_t = \begin{cases}
q^b_t, & P^b_t > P^b_{t-1} \quad (\text{买一价上跳，激进限价买单建立新档位}) \\
q^b_t - q^b_{t-1}, & P^b_t = P^b_{t-1} \quad (\text{买一价维持，净挂单增加或撤单扣减}) \\
-q^b_{t-1}, & P^b_t < P^b_{t-1} \quad (\text{买一价下破，原买档被市价卖单全部吃穿})
\end{cases}
$$

  - 卖方订单流驱动项 $I^a_t$ 对称推导：

$$
I^a_t = \begin{cases}
q^a_t, & P^a_t < P^a_{t-1} \quad (\text{卖一价下跳，激进限价卖单下压建立新档位}) \\
q^a_t - q^a_{t-1}, & P^a_t = P^a_{t-1} \quad (\text{卖一价维持，卖方净挂单增加}) \\
-q^a_{t-1}, & P^a_t > P^a_{t-1} \quad (\text{卖一价抬升，原卖档被市价买单全部吃空})
\end{cases}
$$

  - 单步 L1 OFI 最终定义为双边净通量差：

$$
\text{OFI}_t = I^b_t - I^a_t
$$

- **[引导句锚点 1]**：如下状态判定拓扑所示，OFI 捕捉了盘口价格跳升、维持与破位的所有微观事件流：

```mermaid
flowchart TD
    PrevSnap["盘口快照 t-1: (P_prev^b, q_prev^b), (P_prev^a, q_prev^a)"] --> Event["微观订单事件: 挂单 / 吃单 / 撤单"]
    Event --> CurrSnap["盘口快照 t: (P_curr^b, q_curr^b), (P_curr^a, q_curr^a)"]
    CurrSnap --> CheckBid["买方驱动项 I^b_t: 比较 P_curr^b 与 P_prev^b"]
    CurrSnap --> CheckAsk["卖方驱动项 I^a_t: 比较 P_curr^a 与 P_prev^a"]
    CheckBid --> CalcOFI["OFI_t = I^b_t - I^a_t"]
    CheckAsk --> CalcOFI
    CalcOFI --> ImpactLinear["线性价格冲击拟合: ΔP_mid = β * OFI_t + ε"]
```

- **[架构要点拆解 1]**：
  - **[1. 状态覆盖完备]**：三态分支数学上完全覆盖了离散订单簿所有可能的价格变动组合，无任何遗漏状态。
  - **[2. 物理量守恒]**：买盘新增与卖盘撤销具有同等推高价格的物理功效，OFI 将二者在代数上统一整合为正向驱动力。
  - **[3. 高信噪比]**：相比主动买入额减主动卖出额（Net Taker Volume），OFI 额外计入了被动挂撤单的意图信息，大幅提升了预测能力。

- **[中间价冲击与 Kyle's Lambda 线性关系]**：

$$
\Delta P^{\text{mid}}_t = P^{\text{mid}}_t - P^{\text{mid}}_{t-1} = \beta \cdot \text{OFI}_t + \epsilon_t
$$

- **[变量定义域与物理意义]**：
  - $\beta > 0$：微观 Kyle's Lambda，反映每单位净订单流推动价格跳跃的基点数。在小市值或流动性差的标的上，$\beta$ 极大。

- **[多档深度扩展 Multi-Level OFI 与指数衰减]**：
  - 针对 Level 2 至 Level 10 盘口，引入距离衰减系数 $\alpha > 0$：

$$
\text{OFI}^{(\text{Multi})}_t = \sum_{k=1}^K \exp(-\alpha (k - 1)) \cdot \text{OFI}^{(k)}_t
$$

::: {.callout-warning}
### 投决会风控红线与认知陷阱
- **[因子半衰期极短]**：单步 OFI 的预测能量呈指数级衰减，其有效预测半衰期通常仅有 100 毫秒至数秒。严禁在日线或小时线低频级别直接套用单步高频 OFI。
- **[深档虚假挂单污染]**：未加权的 Multi-Level OFI 极易被买五/卖五以外的远端虚假挂单操纵（Spoofing），必须对 3 档以外的深度赋予强烈的指数衰减权重。
:::

【思考题】
Q1: 为什么 OFI 相比于单纯的“主动买入成交量减主动卖出成交量”（净主动买入额 Net Taker Volume），能够提供信噪比更高、更纯净的短期价格预测能力？请从做市商被动撤单与盘口新挂单事件的信息含量角度推导。
Q2: 为什么在实盘高频量化交易中，单步 OFI 对价格变化的解释力在毫秒至秒级极强（R^2 常常达到 50%~70%），但随着预测时间窗口拉长（如 5分钟、15分钟），其预测能力呈现指数级衰减？做市商如何利用 OFI 衰减半衰期设定库存管理风控？
Q3: 在跨多档盘口（Level 1 至 Level 10）拓展 Multi-Level OFI 时，为什么越深的档位（如买五、买十）需要赋予越大的权重衰减（如指数衰减 exp(-alpha * k)）？深档挂单中存在大量“虚假挂单（Ghost Orders / Spoofing）”会对未加权的多档 OFI 造成怎样的污染？

【评分与通过规则】
- 代码 4 分：
  1. 准确实现单步 L1 OFI 驱动项分支判定 (1分)；
  2. 准确批量计算高频分笔 Tick 序列 OFI 并拓展多档加权 OFI (1.5分)；
  3. 准确实现 OFI 与中间价变动的价格冲击回归估计与 R^2 评估 (1.5分)；
- Q1~Q3 问答题各 2 分（共 6 分）：必须深入说明微观订单流信息含量与半衰期因果链；
- 通过线：8/10 分。一票否决：买卖驱动项逻辑反转、OFI 符号倒置、删除断言。


In [ ]:
from typing import Dict, List, Tuple, Optional
import numpy as np
import pandas as pd


def calculate_ofi(
    b_price_prev: float,
    b_size_prev: int,
    b_price_curr: float,
    b_size_curr: int,
    a_price_prev: float,
    a_size_prev: int,
    a_price_curr: float,
    a_size_curr: int,
) -> float:
    """
    计算单个 Tick 步长的 Level 1 订单流不平衡 (OFI):
        OFI = I_Bid - I_Ask
    """
    # TODO: 实现 Cont-Kukanov-Stoikov (2014) 单步 OFI 计算逻辑
    pass


def calculate_tick_ofi_series(tick_df: pd.DataFrame) -> pd.Series:
    """
    从连续的 Tick 快照数据集中批量计算单步 OFI 序列:
    输入 tick_df 必须包含列: ['b_price', 'b_size', 'a_price', 'a_size']
    第 0 个时刻因为没有前值，OFI 记为 0.0。
    """
    # TODO: 滚动遍历计算整个时序的 OFI 序列
    pass


def calculate_multi_level_ofi(
    b_prices_prev: List[float],
    b_sizes_prev: List[int],
    b_prices_curr: List[float],
    b_sizes_curr: List[int],
    a_prices_prev: List[float],
    a_sizes_prev: List[int],
    a_prices_curr: List[float],
    a_sizes_curr: List[int],
    decay: float = 0.5,
) -> float:
    """
    计算跨多档深度 (Multi-Level) 加权订单流不平衡:
    对于档位 k = 0, 1, ..., K-1:
        权重 w_k = exp(-decay * k)
        OFI_k = calculate_ofi(...)
        Multi_OFI = sum(w_k * OFI_k)
    """
    # TODO: 对各档位按指数衰减加权求和
    pass


def estimate_ofi_price_impact(ofi_series: pd.Series, mid_price_series: pd.Series) -> Dict[str, float]:
    """
    估计 OFI 对中间价变化的一阶价格冲击系数与预测拟合度:
        d_mid_t = mid_price_series_t - mid_price_series_{t-1}
        拟合线性模型: d_mid = beta * ofi + epsilon
    返回:
        {
            "correlation": Pearson 相关系数,
            "beta": 价格冲击弹性系数 (OLS 斜率),
            "r_squared": 拟合优度 R^2
        }
    """
    # TODO: 计算差分对齐后的一阶回归斜率、相关系数与 R^2
    pass


if __name__ == "__main__":
    # ---------------- 基础测试 1: 基础单步 OFI 计算 ----------------
    # 场景: 买一价提升 (10.0 -> 10.1, size 100), 卖一价不变 (10.2 -> 10.2, size 50 -> 40)
    # I_bid = 100, I_ask = 40 - 50 = -10, OFI = 100 - (-10) = 110.0
    ofi = calculate_ofi(10.0, 80, 10.1, 100, 10.2, 50, 10.2, 40)
    assert abs(ofi - 110.0) < 1e-4, f"Expected 110.0, got {ofi}"

    # 场景 2: 买一价下跌 (10.1 -> 10.0, 原挂单被吃), 卖一价下跌 (10.2 -> 10.15, 卖方压盘)
    # I_bid = -100, I_ask = 60, OFI = -100 - 60 = -160.0
    ofi2 = calculate_ofi(10.1, 100, 10.0, 70, 10.2, 50, 10.15, 60)
    assert abs(ofi2 - (-160.0)) < 1e-4, f"Expected -160.0, got {ofi2}"

    # ---------------- 基础测试 2: 多档 Multi-Level OFI ----------------
    # 2 档盘口示例
    bp_prev = [10.0, 9.9]
    bs_prev = [100, 200]
    bp_curr = [10.0, 9.9]
    bs_curr = [150, 200]  # L1 买量增加 50, L2 买量未变

    ap_prev = [10.1, 10.2]
    as_prev = [80, 150]
    ap_curr = [10.1, 10.2]
    as_curr = [60, 150]  # L1 卖量减少 20 (被吃20)

    # L1: I_bid = 50, I_ask = -20, OFI_1 = 70.0, 权重 exp(0) = 1.0
    # L2: I_bid = 0, I_ask = 0, OFI_2 = 0.0
    multi_ofi = calculate_multi_level_ofi(
        bp_prev, bs_prev, bp_curr, bs_curr,
        ap_prev, as_prev, ap_curr, as_curr,
        decay=0.5,
    )
    assert abs(multi_ofi - 70.0) < 1e-4, f"Expected 70.0, got {multi_ofi}"

    # ---------------- 基础测试 3: OFI 时序与价格冲击回归 ----------------
    np.random.seed(42)
    n_ticks = 100
    # 生成合成盘口数据: OFI 净买入正向驱动中间价上涨
    synth_ofi = np.random.normal(0.0, 50.0, n_ticks)
    synth_d_mid = 0.002 * synth_ofi + np.random.normal(0.0, 0.02, n_ticks)
    synth_mid = 100.0 + np.cumsum(synth_d_mid)

    s_ofi = pd.Series(synth_ofi)
    s_mid = pd.Series(synth_mid)

    impact = estimate_ofi_price_impact(s_ofi, s_mid)
    assert impact["correlation"] > 0.85, f"OFI 与价格变化应高度正相关, 实际: {impact['correlation']}"
    assert 0.0015 <= impact["beta"] <= 0.0025, f"Beta 冲击系数应接近 0.002, 实际: {impact['beta']}"
    assert impact["r_squared"] > 0.70, f"R^2 应高于 0.70, 实际: {impact['r_squared']}"

    print("Lesson 03 Exercise Passed!")
